# XGBoost Hyperparameter Tuning — I-66 Phase 1

**Purpose:** Grid search over XGBoost hyperparameters using the **validation split only** (2025-01-01 to 2025-06-30).  
The test split (2025-07-01 to 2025-12-31) is **never touched** during tuning.

**Output:** `best_xgb_params.json` — load this in your main model notebook to replace default params.

**Workflow:**
1. Run this notebook once
2. Check `tuning_summary_for_paper.csv` for selected values → update Table in Section 3.3
3. Load `best_xgb_params.json` in your main model notebook
4. Re-run main model → updated Table 2

In [ ]:
import json
import itertools
from pathlib import Path

import numpy as np
import pandas as pd
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# ============================================================
# PATHS
# ============================================================
GOLD_TABLE_PATH = Path(r"..\..\..\Data\c_final_tables\baseline_model_gold_table\I66_phase1_TTI_features_2022_2025_combined.parquet")

OUT_DIR = Path(r"..\..\..\notebooks\modelling\project_1_output\hyperparameter_tuning")
OUT_DIR.mkdir(parents=True, exist_ok=True)

BEST_PARAMS_PATH    = OUT_DIR / "best_xgb_params.json"
TUNING_RESULTS_PATH = OUT_DIR / "tuning_grid_results.csv"

# ============================================================
# SPLIT DATES — must match your main model notebook exactly
# ============================================================
TRAIN_END = pd.Timestamp("2024-12-31 23:59:59", tz="UTC")
VAL_START = pd.Timestamp("2025-01-01 00:00:00", tz="UTC")
VAL_END   = pd.Timestamp("2025-06-30 23:59:59", tz="UTC")
# Test split (2025-07-01 onward) intentionally excluded from this notebook

# ============================================================
# COLUMNS — matches gold table builder exactly
# ============================================================
TS_COL  = "ts_utc"
TMC_COL = "tmc"

# Lag features: tti_lag_1 ... tti_lag_12
_lags     = [f"tti_lag_{k}" for k in range(1, 13)]
# Rolling mean: windows 6 and 12
_rollmean = [f"tti_rolling_mean_{w}" for w in [6, 12]]
# Rolling std: windows 6 and 12
_rollstd  = [f"tti_rolling_standard_deviation_{w}" for w in [6, 12]]
# Calendar (cyclical + raw)
_calendar = [
    "hour_of_day", "minute_of_hour",
    "day_of_week_number", "month_number",
    "is_weekend_flag",
    "sin_hour", "cos_hour",
    "sin_day_of_week", "cos_day_of_week",
    "sin_month", "cos_month",
]
# Momentum
_momentum = ["tti_change_1_step", "tti_absolute_change_1_step", "is_low_confidence_flag"]

FEATURE_COLS = _lags + _rollmean + _rollstd + _calendar + _momentum

# Target columns already pre-built in the gold table
HORIZONS = {
    "5min":  "target_tti_5min_ahead",
    "15min": "target_tti_15min_ahead",
    "30min": "target_tti_30min_ahead",
}

# ============================================================
# GRID SEARCH SPACE
# ============================================================
PARAM_GRID = {
    "n_estimators":  [100, 300, 500],
    "max_depth":     [3, 5, 7],
    "learning_rate": [0.05, 0.1],
}

# Fixed params (not tuned — reported as-is in paper Table)
FIXED_PARAMS = {
    "subsample":        0.8,
    "colsample_bytree": 0.8,
    "objective":        "reg:squarederror",
    "random_state":     42,
    "n_jobs":           -1,
    "verbosity":        0,
    "tree_method":     "hist",
}

print(f"Feature count: {len(FEATURE_COLS)}")
print(f"Features: {FEATURE_COLS}")

Feature count: 30
Features: ['tti_lag_1', 'tti_lag_2', 'tti_lag_3', 'tti_lag_4', 'tti_lag_5', 'tti_lag_6', 'tti_lag_7', 'tti_lag_8', 'tti_lag_9', 'tti_lag_10', 'tti_lag_11', 'tti_lag_12', 'tti_rolling_mean_6', 'tti_rolling_mean_12', 'tti_rolling_standard_deviation_6', 'tti_rolling_standard_deviation_12', 'hour_of_day', 'minute_of_hour', 'day_of_week_number', 'month_number', 'is_weekend_flag', 'sin_hour', 'cos_hour', 'sin_day_of_week', 'cos_day_of_week', 'sin_month', 'cos_month', 'tti_change_1_step', 'tti_absolute_change_1_step', 'is_low_confidence_flag']


In [2]:
# ============================================================
# LOAD AND SPLIT
# ============================================================
print(f"Loading: {GOLD_TABLE_PATH}")
df = pd.read_parquet(GOLD_TABLE_PATH)
print(f"Rows: {len(df):,} | Cols: {len(df.columns)}")

# Ensure UTC timezone
if df[TS_COL].dt.tz is None:
    df[TS_COL] = df[TS_COL].dt.tz_localize("UTC")

train_df = df[df[TS_COL] <= TRAIN_END].copy()
val_df   = df[(df[TS_COL] >= VAL_START) & (df[TS_COL] <= VAL_END)].copy()

print(f"Train: {len(train_df):,} rows  ({train_df[TS_COL].min().date()} to {train_df[TS_COL].max().date()})")
print(f"Val:   {len(val_df):,} rows  ({val_df[TS_COL].min().date()} to {val_df[TS_COL].max().date()})")

# Confirm all feature columns exist
missing_cols = [c for c in FEATURE_COLS if c not in df.columns]
if missing_cols:
    print(f"WARNING — missing columns: {missing_cols}")
else:
    print(f"All {len(FEATURE_COLS)} feature columns confirmed present.")

Loading: ..\..\..\Data\c_final_tables\baseline_model_gold_table\I66_phase1_TTI_features_2022_2025_combined.parquet
Rows: 17,229,104 | Cols: 44
Train: 12,920,705 rows  (2022-01-01 to 2024-12-31)
Val:   2,136,489 rows  (2025-01-01 to 2025-06-30)
All 30 feature columns confirmed present.


In [3]:
# ============================================================
# GRID SEARCH — one pass per horizon, tuned on validation MAE
# ============================================================
all_combinations = list(itertools.product(
    PARAM_GRID["n_estimators"],
    PARAM_GRID["max_depth"],
    PARAM_GRID["learning_rate"],
))
total_fits = len(all_combinations) * len(HORIZONS)
print(f"Combinations: {len(all_combinations)} x {len(HORIZONS)} horizons = {total_fits} total fits\n")

tuning_rows = []
best_params_per_horizon = {}

for horizon_label, target_col in HORIZONS.items():
    print(f"{'='*55}")
    print(f"Horizon: {horizon_label}  |  target: {target_col}")
    print(f"{'='*55}")

    train_clean = train_df.dropna(subset=FEATURE_COLS + [target_col])
    val_clean   = val_df.dropna(subset=FEATURE_COLS + [target_col])

    X_train = train_clean[FEATURE_COLS].values
    y_train = train_clean[target_col].values
    X_val   = val_clean[FEATURE_COLS].values
    y_val   = val_clean[target_col].values

    print(f"  Train samples: {len(X_train):,}  |  Val samples: {len(X_val):,}")

    best_mae   = np.inf
    best_combo = None

    for n_est, max_d, lr in all_combinations:
        model = XGBRegressor(
            n_estimators=n_est,
            max_depth=max_d,
            learning_rate=lr,
            **FIXED_PARAMS
        )
        model.fit(X_train, y_train)
        preds = model.predict(X_val)

        mae  = mean_absolute_error(y_val, preds)
        rmse = np.sqrt(mean_squared_error(y_val, preds))

        tuning_rows.append({
            "horizon":       horizon_label,
            "n_estimators":  n_est,
            "max_depth":     max_d,
            "learning_rate": lr,
            "val_mae":       round(mae, 6),
            "val_rmse":      round(rmse, 6),
        })

        marker = " <-- best so far" if mae < best_mae else ""
        print(f"  n_est={n_est:3d}  depth={max_d}  lr={lr:.3f}  |  MAE={mae:.5f}  RMSE={rmse:.5f}{marker}")

        if mae < best_mae:
            best_mae   = mae
            best_combo = (n_est, max_d, lr)

    best_params_per_horizon[horizon_label] = {
        "n_estimators":  best_combo[0],
        "max_depth":     best_combo[1],
        "learning_rate": best_combo[2],
        "val_mae":       round(best_mae, 6),
    }
    print(f"\n  BEST {horizon_label}: n_estimators={best_combo[0]}, max_depth={best_combo[1]}, lr={best_combo[2]}  (val_MAE={best_mae:.5f})\n")

print("Grid search complete.")

Combinations: 18 x 3 horizons = 54 total fits

Horizon: 5min  |  target: target_tti_5min_ahead
  Train samples: 12,920,705  |  Val samples: 2,136,489
  n_est=100  depth=3  lr=0.050  |  MAE=0.09601  RMSE=0.29768 <-- best so far
  n_est=100  depth=3  lr=0.100  |  MAE=0.09492  RMSE=0.29333 <-- best so far
  n_est=100  depth=5  lr=0.050  |  MAE=0.09350  RMSE=0.28979 <-- best so far
  n_est=100  depth=5  lr=0.100  |  MAE=0.09300  RMSE=0.28761 <-- best so far
  n_est=100  depth=7  lr=0.050  |  MAE=0.09203  RMSE=0.28565 <-- best so far
  n_est=100  depth=7  lr=0.100  |  MAE=0.09193  RMSE=0.28543 <-- best so far
  n_est=300  depth=3  lr=0.050  |  MAE=0.09441  RMSE=0.29147
  n_est=300  depth=3  lr=0.100  |  MAE=0.09409  RMSE=0.28916
  n_est=300  depth=5  lr=0.050  |  MAE=0.09256  RMSE=0.28672
  n_est=300  depth=5  lr=0.100  |  MAE=0.09251  RMSE=0.28627
  n_est=300  depth=7  lr=0.050  |  MAE=0.09159  RMSE=0.28439 <-- best so far
  n_est=300  depth=7  lr=0.100  |  MAE=0.09183  RMSE=0.28541
  n_es

In [4]:
# ============================================================
# SAVE OUTPUTS
# ============================================================

# 1) Full grid results
tuning_df = pd.DataFrame(tuning_rows)
tuning_df.to_csv(TUNING_RESULTS_PATH, index=False)
print(f"Full grid results: {TUNING_RESULTS_PATH}")

# 2) Best params JSON
output = {
    "description": "Best XGBoost params per horizon, selected on validation MAE (2025-01-01 to 2025-06-30)",
    "fixed_params": FIXED_PARAMS,
    "best_params_per_horizon": best_params_per_horizon,
}
with open(BEST_PARAMS_PATH, "w") as f:
    json.dump(output, f, indent=2)
print(f"Best params JSON:  {BEST_PARAMS_PATH}")

# 3) Paper summary table
summary_rows = []
for horizon_label, params in best_params_per_horizon.items():
    summary_rows.append({
        "Horizon":           horizon_label,
        "n_estimators":      params["n_estimators"],
        "max_depth":         params["max_depth"],
        "learning_rate":     params["learning_rate"],
        "subsample":         FIXED_PARAMS["subsample"],
        "colsample_bytree":  FIXED_PARAMS["colsample_bytree"],
        "Val MAE":           params["val_mae"],
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUT_DIR / "tuning_summary_for_paper.csv", index=False)

print("\n=== Section 3.3 Table Values ===")
print(summary_df.to_string(index=False))

Full grid results: ..\..\..\notebooks\modelling\model_outputs\Phase_1_model\hyperparameter_tuning\tuning_grid_results.csv
Best params JSON:  ..\..\..\notebooks\modelling\model_outputs\Phase_1_model\hyperparameter_tuning\best_xgb_params.json

=== Section 3.3 Table Values ===
Horizon  n_estimators  max_depth  learning_rate  subsample  colsample_bytree  Val MAE
   5min           500          7           0.05        0.8               0.8 0.091553
  15min           500          7           0.05        0.8               0.8 0.118458
  30min           500          7           0.10        0.8               0.8 0.140706


## How to use in your main model notebook

Add this near the top of your main model notebook, replacing any hardcoded XGBoost params:

```python
import json

with open(r"..\model_outputs\Phase_1_model\hyperparameter_tuning\best_xgb_params.json") as f:
    tuning = json.load(f)

# When building the model for each horizon:
params = tuning["best_params_per_horizon"]["5min"]  # or "15min", "30min"
model = XGBRegressor(
    n_estimators     = params["n_estimators"],
    max_depth        = params["max_depth"],
    learning_rate    = params["learning_rate"],
    subsample        = 1.0,
    colsample_bytree = 1.0,
    objective        = "reg:squarederror",
    random_state     = 42,
    n_jobs           = -1,
)
```